# Wasserstein GANs (WGAN)

Wasserstein GANs introduce the following changes to vanilla GANs:

- Critic: The Discriminator in GANs turns into a Critic in WGAN. The role of the critic is similar to the discriminator, but its output is not bound as in a GAN (the critic does not use a sigmoid activation at its output). It can also be seen as a score of how real an image is, but it is harder to interpret as there is no maximum or minimum value.

- Error function: WGANs use the Wasserstein distance to measure the difference between the distributions of the real and fake data.

    - **Critic (Discriminator) Loss:**
        $$ L_{\text{critic}} = -\mathbb{E}_{x \sim p_{\text{real}}}[D(x)] + \mathbb{E}_{z \sim p_z}[D(G(z))] $$


    - **Generator Loss:**
        $$ L_{\text{generator}} = -\mathbb{E}_{z \sim p_z}[D(G(z))] $$

- Gradient Penalty: the gradient penalty enforces the 1-Lipschitz constraint, which requires the gradient norm of the critic to not be larger than 1. The original WGAN paper used weight clipping to enforce this. In WGAN-GP, the gradient penalty encourages the gradient norm to be close to 1 by penalizing deviations from this value.

- Training cycle: In a WGAN, the critic is trained for $n$ iterations (typically 5 or more) for each update of the generator.

These modifications aim to increase the stability in training. WGANs and WGAN-GP have been shown to provide more stable and robust training compared to vanilla GANs, both in theory and in empirical studies. However, they do not guarantee perfect stability in all settings.

**Refs**

- Martin Arjovsky, Soumith Chintala, and Léon Bottou (2017) - [Wasserstein GAN](https://arxiv.org/abs/1701.07875)

- Ishaan Gulrajani, Faruk Ahmed, Martin Arjovsky, Vincent Dumoulin, Aaron Courville (2017) - [Improved Training of Wasserstein GANs](https://proceedings.neurips.cc/paper_files/paper/2017/file/892c3b1c6dccd52936e27cbd0ff683d6-Paper.pdf)


    





In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

import torchinfo

import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib.patches import Rectangle

import numpy as np
import time

import os
import pathlib
from PIL import Image
import skimage

# importing a module with utilities for displaying stats and data
import sys
sys.path.insert(1, 'util')
import vcpi_util

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
def show_images(rows, cols, images):

    width= 2 * rows
    height= 2 * cols

    f, axes= plt.subplots(rows,cols,figsize=(height,width))
    fig=plt.figure()

    for a in range(rows*cols):

        axes.ravel()[a].imshow(np.clip(np.transpose(images[a].numpy(),(1,2,0))*0.5 + 0.5,0,1), cmap=plt.cm.gray)
        axes.ravel()[a].axis('off')
    fig.tight_layout()    
    plt.show()  

## Configuration

In [ ]:
LATENT_SPACE_DIM = 16
BATCH_SIZE = 32

EPOCHS = 30

In [ ]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize(mean=[0.5], std=[0.5])]) 



train_set = torchvision.datasets.MNIST(root='./data', train=True,
                                        download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE,
                                          shuffle=True)

test_set = torchvision.datasets.MNIST(root='./data', train=False,
                                       download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=BATCH_SIZE,
                                         shuffle=False)

In [ ]:
images, targets = next(iter(train_loader))

# these are the transformed images
vcpi_util.show_transformed_images(3,8, train_set, train_set.classes) 

## Generator        

The WGAN generator is similar to a GAN generator

In [ ]:
class Generator(nn.Module):

    def __init__(self, latent_dim = 16):
        super().__init__()

        self.model = nn.Sequential(
            # Input: (batch_size, latent_dim, 1, 1)
            nn.ConvTranspose2d(latent_dim, 128, kernel_size=7, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            nn.ConvTranspose2d(64, 1, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        # z shape: (batch_size, latent_dim)
        z = z.view(z.size(0), z.size(1), 1, 1)
        img = self.model(z)
        return img  # shape: (batch_size, 1, 28, 28)

In [ ]:
G = Generator(LATENT_SPACE_DIM)
G.to(device)

torchinfo.summary(G, input_size = (BATCH_SIZE, LATENT_SPACE_DIM,1,1),
                                     col_names=["kernel_size", "input_size", "output_size", "num_params", "mult_adds"])

In [ ]:
noise = torch.randn(BATCH_SIZE, LATENT_SPACE_DIM, 1, 1, device=device)

out = G(noise)
print(out.shape)

plt.imshow(np.transpose(out[0].detach().cpu().numpy() * 0.5 + 0.5, (1,2,0)), cmap='gray')

## Critic

The critic replaces the Discriminator in a GAN, the main difference being that, while the later has a sigmiod activation function for the output, the Critic produces an unbounded output.

In [ ]:
class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            # Input: (batch_size, 1, 28, 28)
            nn.Conv2d(1, 64, 4, 2, 1, bias=True),   # No batchnorm in first layer
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
C = Critic()
C.to(device)

torchinfo.summary(C, input_size = (BATCH_SIZE, 1,28,28),
                                     col_names=["kernel_size", "input_size", "output_size", "num_params", "mult_adds"])

In [ ]:
optim_C = torch.optim.Adam(C.parameters(), lr=1e-4, betas=(0.0, 0.9))
optim_G = torch.optim.Adam(G.parameters(), lr=1e-4, betas=(0.0, 0.9))

## Training 

Training is a two step procedure, as in a vanilla GAN. 

The main differences are:

- Critic is trained for $n$ iterations per batch
- The error function is now based on the Wasserstein distance
- A gradient penalty term to discourage gradients with norms different from 1.


In [ ]:
def gradient_penalty(D, real, fake):
    alpha = torch.rand(real.size(0), 1, 1, 1, device=device)
    interpolates = alpha * real + (1 - alpha) * fake
    interpolates.requires_grad_(True)
    d_interpolates = D(interpolates)
    fake_out = torch.ones(d_interpolates.size(), device=device)
    gradients = torch.autograd.grad(
        outputs=d_interpolates, inputs=interpolates,
        grad_outputs=fake_out, create_graph=True, retain_graph=True, only_inputs=True
    )[0]
    gradients = gradients.view(gradients.size(0), -1)
    return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

In [ ]:
EPOCHS = 30
iter_critic = 5
lambda_gp = 10

C.train()
G.train()
C.to(device)
G.to(device)

latent_vis = torch.randn(64, LATENT_SPACE_DIM, 1, 1, device=device)



for epoch in range(EPOCHS):
    for i, (digits, _) in enumerate(train_loader):
        digits = digits.to(device)
        for _ in range(iter_critic):
            optim_C.zero_grad()
            out_C_real = C(digits)
            latent = torch.randn(len(digits), LATENT_SPACE_DIM, 1, 1, device=device)
            fakes = G(latent).detach()
            out_C_fakes = C(fakes)
            gp = gradient_penalty(C, digits.detach(), fakes.detach())
            c_loss = -torch.mean(out_C_real) + torch.mean(out_C_fakes) + lambda_gp * gp
            c_loss.backward()
            optim_C.step()

        optim_G.zero_grad()
        latent = torch.randn(len(digits), LATENT_SPACE_DIM, 1, 1, device=device)
        fakes = G(latent)
        out = C(fakes)
        g_loss = -torch.mean(out)
        g_loss.backward()
        optim_G.step()

    # stats
    print('[%d][%d/%d]\tLoss_D: %.4f\tLoss_G: %.4f'
    % (epoch, i, len(train_loader),
    c_loss.item(), g_loss.item()))

    with torch.no_grad():
        fake = G(latent_vis).detach().cpu()
        grid = torchvision.utils.make_grid(fake, padding=2, normalize=True)
    plt.imshow(np.transpose(grid, (1, 2, 0)))
    plt.axis('off')
    plt.show()

    torch.save({
        'epoch': epoch,
        'generator': G.state_dict(),
        'optim_G': optim_G.state_dict(),
        'discriminator': C.state_dict(),
        'optim_D': optim_C.state_dict()
    },
    f'models_gan_mnist/wgan_mnist_{LATENT_SPACE_DIM}_{epoch}.pt')        